# 50 RoBERTa LoRA Pipeline (Parallel-safe)

Runs RoBERTa-base LoRA training, inference, cleaning, and evaluation with isolated paths so it can run in parallel with Gemma jobs.


In [ ]:
# Cell 0: Configure model, paths, and run toggles.
import os
import re
import ast
import json
import textwrap
import subprocess
from datetime import datetime
from pathlib import Path

PROJECT_ROOT = Path('/home/rameyjm7/workspace/llamafactory-gemma-lora')
EXPRESS_ROOT = PROJECT_ROOT / 'express-emotion-recognition'
REGISTRY_PATH = PROJECT_ROOT / 'configs' / 'model_registry.yaml'
ENV_FILE = PROJECT_ROOT / 'env.txt'
ENV_PKL_FILE = PROJECT_ROOT / 'configs' / 'env.pkl'

MODEL_ID = 'longformer_base'

# Distributed training controls.
DISTRIBUTED_TRAIN = True
NPROC_PER_NODE = 2
MASTER_PORT = 29641
GPU_IDS = '0,1'      # GPUs used when DISTRIBUTED_TRAIN=True
SINGLE_GPU_ID = '0'  # GPU used when DISTRIBUTED_TRAIN=False

RUN_TRAIN = True
RUN_INFERENCE = True
RUN_EVAL = True

TRAIN_ROWS = 12000  # use None for full express.csv
MAX_LENGTH = 512
LR = 5e-5
EPOCHS = 2.0
TRAIN_BATCH_SIZE = 16
GRAD_ACCUM = 1

if DISTRIBUTED_TRAIN:
    os.environ['CUDA_VISIBLE_DEVICES'] = GPU_IDS

    visible_ids = [x.strip() for x in str(os.environ.get('CUDA_VISIBLE_DEVICES', '')).split(',') if x.strip()]
    try:
        import torch
        detected_gpus = int(torch.cuda.device_count())
    except Exception:
        detected_gpus = len(visible_ids)

    # Prevent NCCL duplicate-GPU errors by disabling DDP unless 2+ visible devices are confirmed.
    if detected_gpus < 2 or len(set(visible_ids)) < 2:
        print(f'Warning: DDP requested but only {detected_gpus} visible GPU(s); falling back to single-GPU mode.')
        DISTRIBUTED_TRAIN = False
        NPROC_PER_NODE = 1
        if SINGLE_GPU_ID is not None:
            os.environ['CUDA_VISIBLE_DEVICES'] = str(SINGLE_GPU_ID)
    else:
        NPROC_PER_NODE = min(int(NPROC_PER_NODE), detected_gpus)
elif SINGLE_GPU_ID is not None:
    os.environ['CUDA_VISIBLE_DEVICES'] = str(SINGLE_GPU_ID)

MODEL_CACHE_DIR = PROJECT_ROOT / 'outputs' / 'hf_cache' / f'hf_cache_{MODEL_ID}'
RUN_DIR = PROJECT_ROOT / 'outputs' / 'lora_runs' / f'{MODEL_ID}_express_lora'
METRICS_DIR = PROJECT_ROOT / 'outputs' / 'metrics' / MODEL_ID
FIGURES_DIR = PROJECT_ROOT / 'outputs' / 'figures' / MODEL_ID

EVAL_INPUT_PATH = METRICS_DIR / f'{MODEL_ID}_baseline_eval_input.csv'
RAW_OUTPUT_PATH = METRICS_DIR / f'{MODEL_ID}_lora_raw.csv'
CLEAN_OUTPUT_PATH = METRICS_DIR / f'{MODEL_ID}_lora_clean.csv'
TRAIN_PREP_PATH = METRICS_DIR / f'{MODEL_ID}_train_prep_stats.json'
METRICS_TABLE_PATH = PROJECT_ROOT / 'outputs' / 'metrics' / 'all_models_metrics.csv'
FIGURE_PATH = FIGURES_DIR / f'figure2_with_lora_overlay_{MODEL_ID}.png'

for p in [MODEL_CACHE_DIR, RUN_DIR, METRICS_DIR, FIGURES_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('MODEL_ID =', MODEL_ID)
print('RUN_DIR =', RUN_DIR)
print('METRICS_DIR =', METRICS_DIR)
print('CACHE_DIR =', MODEL_CACHE_DIR)
print('DISTRIBUTED_TRAIN =', DISTRIBUTED_TRAIN)
print('NPROC_PER_NODE =', NPROC_PER_NODE)
print('CUDA_VISIBLE_DEVICES =', os.environ.get('CUDA_VISIBLE_DEVICES', '<default>'))






In [ ]:
# Cell 1: Load token, registry info, and command helpers.
import yaml
import pickle


def load_hf_token(env_path: Path, pkl_path: Path) -> str:
    if pkl_path.exists():
        try:
            with pkl_path.open('rb') as f:
                payload = pickle.load(f)
            token = str(payload.get('HF_TOKEN', '')).strip()
            if token:
                return token
        except Exception as exc:
            print(f'Warning: failed to read {pkl_path}: {exc}')

    if not env_path.exists():
        raise FileNotFoundError(f'Env file not found: {env_path}')

    token = ''
    for raw in env_path.read_text(encoding='utf-8').splitlines():
        line = raw.strip()
        if not line or line.startswith('#'):
            continue
        if line.startswith('HF_TOKEN='):
            token = line.split('=', 1)[1].strip().strip('\"').strip("'")
            break

    if not token:
        raise ValueError(f'HF_TOKEN not found or empty in {env_path} or {pkl_path}')

    return token

def run_cmd(cmd, cwd=None, env=None, check=True):
    printable = ' '.join(str(x) for x in cmd)
    print(f'$ {printable}')
    merged_env = os.environ.copy()
    if env:
        merged_env.update(env)

    proc = subprocess.Popen(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        env=merged_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    lines = []
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
        lines.append(line)

    rc = proc.wait()
    out = ''.join(lines)
    if check and rc != 0:
        tail = ''.join(lines[-100:])
        raise RuntimeError(
            f'Command failed ({rc}): {printable}\n'
            f'--- Last output lines ---\n{tail}'
        )
    return out


def parse_eval_metrics(stdout_text):
    metrics = {}
    for key in ['VRate', 'AccL', 'AccV', 'F1V', 'AccV-2']:
        m = re.search(rf'{re.escape(key)}\s*=\s*([0-9]*\.?[0-9]+)', stdout_text)
        if m:
            metrics[key] = float(m.group(1))
    return metrics


HF_TOKEN = load_hf_token(ENV_FILE, ENV_PKL_FILE)
os.environ['HF_TOKEN'] = HF_TOKEN

registry = yaml.safe_load(REGISTRY_PATH.read_text(encoding='utf-8'))['models']
registry_map = {m['id']: m for m in registry}
assert MODEL_ID in registry_map, f'{MODEL_ID} not found in {REGISTRY_PATH}'
MODEL_SPEC = registry_map[MODEL_ID]

MODEL_NAME_OR_PATH = MODEL_SPEC['model_name_or_path']
MODEL_SIZE_B = float(MODEL_SPEC['size_b'])

print('Loaded HF token using', ENV_PKL_FILE, 'with fallback to', ENV_FILE)
print('Model spec:', MODEL_SPEC)




In [ ]:
# Cell 2: Prepare train/eval datasets with isolated output paths.
import pandas as pd

train_source = EXPRESS_ROOT / 'data' / 'express.csv'
eval_source = EXPRESS_ROOT / 'results' / 'longformer.csv'

train_df = pd.read_csv(train_source)
if TRAIN_ROWS is not None:
    train_df = train_df.head(TRAIN_ROWS)
train_df = train_df[['segment', 'labels', 'number_of_labels']].copy()

eval_df = pd.read_csv(eval_source)
eval_df = eval_df[['segment', 'labels', 'number_of_labels']].copy()
eval_df.to_csv(EVAL_INPUT_PATH, index=False)

print('Train rows:', len(train_df), 'from', train_source)
print('Eval rows:', len(eval_df), 'from', eval_source)
print('Saved eval input to', EVAL_INPUT_PATH)

display(train_df.head(2))
display(eval_df.head(2))


In [ ]:
# Cell 3: Train Longformer LoRA (single GPU or DDP via torchrun).
if RUN_TRAIN:
    import torch

    train_script_path = RUN_DIR / 'train_longformer_lora_worker.py'
    train_csv_path = EXPRESS_ROOT / 'data' / 'express.csv'

    worker_script = textwrap.dedent("""
import os
import ast
import json
import argparse
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForMaskedLM, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model


def safe_parse_label_list(raw):
    if isinstance(raw, list):
        return [str(x).strip().lower() for x in raw]
    if not isinstance(raw, str):
        return None
    txt = raw.strip()
    if not txt:
        return None
    try:
        val = ast.literal_eval(txt)
        if isinstance(val, list):
            return [str(x).strip().lower() for x in val]
    except Exception:
        return None
    return None


def build_examples(df, tokenizer, max_length):
    examples = []
    skipped_parse = 0
    skipped_mask_mismatch = 0
    skipped_multi_token = 0

    for row in df.itertuples(index=False):
        segment = str(row.segment)
        labels = safe_parse_label_list(row.labels)
        if labels is None:
            skipped_parse += 1
            continue

        enc = tokenizer(segment, truncation=True, max_length=max_length)
        input_ids = enc['input_ids']
        attention_mask = enc['attention_mask']
        mask_positions = [i for i, tok in enumerate(input_ids) if tok == tokenizer.mask_token_id]

        if len(mask_positions) != len(labels):
            skipped_mask_mismatch += 1
            continue

        target_ids = []
        valid = True
        for lab in labels:
            # Longformer tokenization is space-sensitive; try word-start form first.
            ids = tokenizer(' ' + lab, add_special_tokens=False)['input_ids']
            if len(ids) != 1:
                ids = tokenizer(lab, add_special_tokens=False)['input_ids']
            if len(ids) != 1:
                valid = False
                break
            target_ids.append(ids[0])

        if not valid:
            skipped_multi_token += 1
            continue

        label_ids = [-100] * len(input_ids)
        for pos, tok in zip(mask_positions, target_ids):
            label_ids[pos] = tok

        examples.append(
            {
                'input_ids': input_ids,
                'attention_mask': attention_mask,
                'labels': label_ids,
            }
        )

    stats = {
        'total_rows': int(len(df)),
        'kept_rows': int(len(examples)),
        'skipped_parse': int(skipped_parse),
        'skipped_mask_mismatch': int(skipped_mask_mismatch),
        'skipped_multi_token_labels': int(skipped_multi_token),
        'max_length': int(max_length),
    }
    return examples, stats


class ListDataset(Dataset):
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx]


def collate_builder(tokenizer):
    def collate_fn(features):
        batch = tokenizer.pad(
            {
                'input_ids': [f['input_ids'] for f in features],
                'attention_mask': [f['attention_mask'] for f in features],
            },
            padding=True,
            return_tensors='pt',
        )
        max_len = batch['input_ids'].shape[1]
        labels = []
        for f in features:
            padded = f['labels'] + ([-100] * (max_len - len(f['labels'])))
            labels.append(padded)
        batch['labels'] = torch.tensor(labels, dtype=torch.long)
        return batch

    return collate_fn


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--model_name_or_path', required=True)
    parser.add_argument('--train_source', required=True)
    parser.add_argument('--run_dir', required=True)
    parser.add_argument('--cache_dir', required=True)
    parser.add_argument('--train_prep_path', required=True)
    parser.add_argument('--train_rows', type=int, default=-1)
    parser.add_argument('--max_length', type=int, default=256)
    parser.add_argument('--learning_rate', type=float, default=5e-5)
    parser.add_argument('--epochs', type=float, default=2.0)
    parser.add_argument('--batch_size', type=int, default=16)
    parser.add_argument('--grad_accum', type=int, default=1)
    parser.add_argument('--model_id', required=True)
    args = parser.parse_args()

    hf_token = os.environ.get('HF_TOKEN')
    if not hf_token:
        raise RuntimeError('HF_TOKEN is required in environment for training.')

    run_dir = Path(args.run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(
        args.model_name_or_path,
        token=hf_token,
        cache_dir=args.cache_dir,
        use_fast=True,
    )
    if tokenizer.mask_token_id is None:
        raise ValueError('Tokenizer has no mask token; cannot run MLM training.')

    train_df = pd.read_csv(args.train_source)
    if args.train_rows is not None and args.train_rows >= 0:
        train_df = train_df.head(args.train_rows)
    train_df = train_df[['segment', 'labels', 'number_of_labels']].copy()

    examples, prep_stats = build_examples(train_df, tokenizer, args.max_length)

    rank = int(os.environ.get('RANK', '0'))
    world_size = int(os.environ.get('WORLD_SIZE', '1'))
    if rank == 0:
        Path(args.train_prep_path).write_text(json.dumps(prep_stats, indent=2), encoding='utf-8')
        print('Training prep stats:', prep_stats)

    if not examples:
        raise RuntimeError('No valid training examples were built.')

    train_dataset = ListDataset(examples)
    collate_fn = collate_builder(tokenizer)

    base_model = AutoModelForMaskedLM.from_pretrained(
        args.model_name_or_path,
        token=hf_token,
        cache_dir=args.cache_dir,
    )

    lora_cfg = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias='none',
        target_modules=['query', 'key', 'value'],
    )
    model = get_peft_model(base_model, lora_cfg)
    if rank == 0:
        model.print_trainable_parameters()

    use_cuda = torch.cuda.is_available()
    use_bf16 = bool(use_cuda and torch.cuda.is_bf16_supported())
    use_fp16 = bool(use_cuda and not use_bf16)

    trainer_args = TrainingArguments(
        output_dir=str(run_dir / 'trainer_state'),
        per_device_train_batch_size=args.batch_size,
        gradient_accumulation_steps=args.grad_accum,
        learning_rate=args.learning_rate,
        num_train_epochs=args.epochs,
        logging_steps=20,
        save_strategy='epoch',
        remove_unused_columns=False,
        fp16=use_fp16,
        bf16=use_bf16,
        ddp_find_unused_parameters=False,
        report_to=[],
    )

    trainer = Trainer(
        model=model,
        args=trainer_args,
        train_dataset=train_dataset,
        data_collator=collate_fn,
    )

    train_out = trainer.train()

    if trainer.is_world_process_zero():
        trainer.model.save_pretrained(str(run_dir))
        tokenizer.save_pretrained(str(run_dir))
        summary = {
            'train_runtime': float(getattr(train_out, 'metrics', {}).get('train_runtime', 0.0)),
            'train_samples': int(len(train_dataset)),
            'timestamp_utc': datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z'),
            'model_id': args.model_id,
            'model_name_or_path': args.model_name_or_path,
            'world_size': world_size,
        }
        (run_dir / 'training_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
        print('Saved LoRA adapter to:', run_dir)


if __name__ == '__main__':
    main()
    """).strip() + '\n'

    train_script_path.write_text(worker_script, encoding='utf-8')
    print('Wrote training worker script to:', train_script_path)

    train_rows_arg = -1 if TRAIN_ROWS is None else int(TRAIN_ROWS)

    common_args = [
        '--model_name_or_path', str(MODEL_NAME_OR_PATH),
        '--train_source', str(train_csv_path),
        '--run_dir', str(RUN_DIR),
        '--cache_dir', str(MODEL_CACHE_DIR),
        '--train_prep_path', str(TRAIN_PREP_PATH),
        '--train_rows', str(train_rows_arg),
        '--max_length', str(MAX_LENGTH),
        '--learning_rate', str(LR),
        '--epochs', str(EPOCHS),
        '--batch_size', str(TRAIN_BATCH_SIZE),
        '--grad_accum', str(GRAD_ACCUM),
        '--model_id', str(MODEL_ID),
    ]

    env = {
        'HF_TOKEN': HF_TOKEN,
        'TOKENIZERS_PARALLELISM': 'false',
    }

    if DISTRIBUTED_TRAIN:
        if int(NPROC_PER_NODE) < 2:
            print('DDP disabled because effective NPROC_PER_NODE < 2; using single process.')
            DISTRIBUTED_TRAIN = False
        cmd = [
            'torchrun',
            '--standalone',
            f'--nproc_per_node={NPROC_PER_NODE}',
            f'--master_port={MASTER_PORT}',
            str(train_script_path),
            *common_args,
        ]
    if not DISTRIBUTED_TRAIN:
        cmd = ['python', str(train_script_path), *common_args]

    _ = run_cmd(cmd, cwd=PROJECT_ROOT, env=env, check=True)
else:
    print('Training skipped (RUN_TRAIN=False).')




In [ ]:
# Cell 4: Run Longformer LoRA inference on baseline-comparable rows.
if RUN_INFERENCE:
    import torch
    from tqdm import tqdm
    from transformers import AutoTokenizer, AutoModelForMaskedLM
    from peft import PeftModel

    eval_df = pd.read_csv(EVAL_INPUT_PATH)

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME_OR_PATH,
        token=HF_TOKEN,
        cache_dir=str(MODEL_CACHE_DIR),
        use_fast=True,
    )
    base_model = AutoModelForMaskedLM.from_pretrained(
        MODEL_NAME_OR_PATH,
        token=HF_TOKEN,
        cache_dir=str(MODEL_CACHE_DIR),
    )
    model = PeftModel.from_pretrained(base_model, str(RUN_DIR))

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = model.to(device)
    model.eval()

    mask_token = tokenizer.mask_token
    if mask_token is None:
        raise ValueError('Tokenizer has no mask token; cannot run MLM inference.')

    def normalize_pred_token(text):
        txt = re.sub(r'\s+', ' ', str(text)).strip().lower()
        return txt

    def predict_mask_list(segment, num_labels):
        working_text = str(segment)
        target_n = int(num_labels)
        preds = []

        for _ in range(target_n):
            if mask_token not in working_text:
                break

            enc = tokenizer(
                working_text,
                return_tensors='pt',
                truncation=True,
                max_length=MAX_LENGTH,
            )
            enc = {k: v.to(device) for k, v in enc.items()}

            mask_positions = (enc['input_ids'][0] == tokenizer.mask_token_id).nonzero(as_tuple=False)
            if len(mask_positions) == 0:
                break

            pos = int(mask_positions[0].item())
            with torch.inference_mode():
                logits = model(**enc).logits[0, pos]
            pred_id = int(torch.argmax(logits).item())
            pred_word = normalize_pred_token(tokenizer.decode([pred_id], skip_special_tokens=True))
            if not pred_word:
                pred_word = 'neutral'

            preds.append(pred_word)
            working_text = working_text.replace(mask_token, pred_word, 1)

        if len(preds) != target_n:
            return 'INVALID OUTPUT'
        return str(preds)

    outputs = []
    for row in tqdm(eval_df.itertuples(index=False), total=len(eval_df), desc='Longformer LoRA inference'):
        outputs.append(predict_mask_list(row.segment, row.number_of_labels))

    raw_df = eval_df.copy()
    raw_df['output'] = outputs
    raw_df.to_csv(RAW_OUTPUT_PATH, index=False)

    print('Saved raw outputs to:', RAW_OUTPUT_PATH)
    print('Rows:', len(raw_df))
else:
    print('Inference skipped (RUN_INFERENCE=False).')


In [ ]:
# Cell 5: Clean and evaluate outputs with existing project scripts.
lora_metrics = {}
if RUN_EVAL:
    _ = run_cmd(
        [
            'python', str(EXPRESS_ROOT / 'src' / 'evaluation' / 'result-cleaning.py'),
            '--input', str(RAW_OUTPUT_PATH),
            '--output', str(CLEAN_OUTPUT_PATH),
        ],
        cwd=PROJECT_ROOT,
        check=True,
    )

    eval_stdout = run_cmd(
        [
            'python', str(EXPRESS_ROOT / 'src' / 'evaluation' / 'result-evaluation.py'),
            '--result_path', str(CLEAN_OUTPUT_PATH),
            '--breakdowns_path', str(EXPRESS_ROOT / 'data' / 'lexicon-decomposition.csv'),
        ],
        cwd=PROJECT_ROOT,
        check=True,
    )

    lora_metrics = parse_eval_metrics(eval_stdout)
    print('LoRA metrics:', lora_metrics)
else:
    print('Evaluation skipped (RUN_EVAL=False).')


In [ ]:
from datetime import datetime, timezone
# Cell 6: Upsert shared metrics table for cross-model comparison.
import pandas as pd

row = {
    'timestamp_utc': datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z'),
    'model_id': MODEL_ID,
    'family': MODEL_SPEC['family'],
    'model_name_or_path': MODEL_NAME_OR_PATH,
    'size_b': MODEL_SIZE_B,
    'pipeline': MODEL_SPEC['pipeline'],
    'is_lora': True,
    'v_rate': lora_metrics.get('VRate'),
    'accl': lora_metrics.get('AccL'),
    'accv': lora_metrics.get('AccV'),
    'f1v': lora_metrics.get('F1V'),
    'accv2': lora_metrics.get('AccV-2'),
    'clean_output_path': str(CLEAN_OUTPUT_PATH),
    'run_dir': str(RUN_DIR),
}

if METRICS_TABLE_PATH.exists():
    metrics_df = pd.read_csv(METRICS_TABLE_PATH)
else:
    metrics_df = pd.DataFrame()

if not metrics_df.empty and {'model_id', 'is_lora'}.issubset(metrics_df.columns):
    metrics_df = metrics_df[~((metrics_df['model_id'] == MODEL_ID) & (metrics_df['is_lora'].astype(str).str.lower() == 'true'))]

metrics_df = pd.concat([metrics_df, pd.DataFrame([row])], ignore_index=True)
metrics_df.to_csv(METRICS_TABLE_PATH, index=False)

print('Updated metrics table:', METRICS_TABLE_PATH)
display(metrics_df.tail(10))


In [ ]:
# Cell 7: Plot Figure-2 baseline points with Longformer LoRA overlay.
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

lex_df = pd.read_csv(EXPRESS_ROOT / 'data' / 'lexicon-decomposition.csv')
vec_map = {str(r['word']).lower(): tuple(int(v) for v in r.iloc[1:].tolist()) for _, r in lex_df.iterrows()}
zero_vec = tuple([0] * 10)

_parse_cache = {}
def parse_list_cached(raw):
    if isinstance(raw, list):
        return [str(x).strip().lower() for x in raw]
    if not isinstance(raw, str):
        return None
    s = raw.strip()
    if s == 'INVALID OUTPUT':
        return None
    if s in _parse_cache:
        return _parse_cache[s]
    try:
        v = ast.literal_eval(s)
        parsed = [str(x).strip().lower() for x in v] if isinstance(v, list) else None
    except Exception:
        parsed = None
    _parse_cache[s] = parsed
    return parsed


def compute_accv(csv_path):
    df = pd.read_csv(csv_path, usecols=['labels', 'output_formatted', 'number_of_labels'])
    total_masks = int(df['number_of_labels'].sum())
    matches = 0
    for raw_labels, raw_preds in zip(df['labels'].tolist(), df['output_formatted'].tolist()):
        labels = parse_list_cached(raw_labels)
        preds = parse_list_cached(raw_preds)
        if labels is None or preds is None:
            continue
        for t, p in zip(labels, preds):
            matches += int(vec_map.get(t, zero_vec) == vec_map.get(p, zero_vec))
    return (matches / total_masks) if total_masks > 0 else float('nan')

baseline_models = [
    {'label': 'Flan-T5-large', 'family': 'Flan-T5', 'size_b': 0.78, 'file': 'flan_t5_large.csv'},
    {'label': 'Flan-T5-xl', 'family': 'Flan-T5', 'size_b': 3.0, 'file': 'flan_t5_xl.csv'},
    {'label': 'Flan-T5-xxl', 'family': 'Flan-T5', 'size_b': 11.0, 'file': 'flan_t5_xxl.csv'},
    {'label': 'Gpt-3.5-turbo', 'family': 'GPT', 'size_b': 175.0, 'file': 'gpt_35_turbo_0125.csv'},
    {'label': 'Gpt-4o', 'family': 'GPT', 'size_b': 1750.0, 'file': 'gpt_4o.csv'},
    {'label': 'Gemma-2-2B-it', 'family': 'Gemma2', 'size_b': 2.0, 'file': 'gemma2_2B_it.csv'},
    {'label': 'Gemma-2-9B-it', 'family': 'Gemma2', 'size_b': 9.0, 'file': 'gemma2_9B_it.csv'},
    {'label': 'Gemma-2-27B-it', 'family': 'Gemma2', 'size_b': 27.0, 'file': 'gemma2_27B_it.csv'},
    {'label': 'Llama3.1-8B-instruct', 'family': 'Llama3.1', 'size_b': 8.0, 'file': 'llama3.1_8B_instruct.csv'},
    {'label': 'Llama3.1-70B-instruct', 'family': 'Llama3.1', 'size_b': 70.0, 'file': 'llama3.1_70B_instruct.csv'},
    {'label': 'Longformer-base', 'family': 'Longformer', 'size_b': 0.149, 'file': 'longformer.csv'},
    {'label': 'Mental-Longformer-base', 'family': 'Longformer', 'size_b': 0.149, 'file': 'mental-longformer.csv'},
    {'label': 'Roberta-base', 'family': 'Roberta', 'size_b': 0.125, 'file': 'roberta-base.csv'},
    {'label': 'Mental-Roberta-base', 'family': 'Roberta', 'size_b': 0.125, 'file': 'mental-roberta-base.csv'},
]

rows = []
for m in baseline_models:
    p = EXPRESS_ROOT / 'results' / m['file']
    rows.append({**m, 'accv': compute_accv(p)})
fig_df = pd.DataFrame(rows)

if CLEAN_OUTPUT_PATH.exists():
    lora_accv = compute_accv(CLEAN_OUTPUT_PATH)
else:
    lora_accv = None

sns.set_theme(style='whitegrid')
family_colors = {
    'Flan-T5': '#4C72B0',
    'GPT': '#DD8452',
    'Gemma2': '#55A868',
    'Llama3.1': '#C44E52',
    'Longformer': '#8172B2',
    'Roberta': '#937860',
}
family_markers = {
    'Flan-T5': 'D',
    'GPT': 'P',
    'Gemma2': '^',
    'Llama3.1': 'X',
    'Longformer': 's',
    'Roberta': 'o',
}

plt.figure(figsize=(14, 6))
for fam in ['Flan-T5', 'GPT', 'Gemma2', 'Llama3.1', 'Longformer', 'Roberta']:
    sub = fig_df[fig_df['family'] == fam].sort_values('size_b')
    if sub.empty:
        continue
    plt.plot(
        sub['size_b'], sub['accv'],
        marker=family_markers[fam],
        color=family_colors[fam],
        linestyle='--',
        linewidth=1.25,
        markersize=8,
        label=fam,
    )

if lora_accv is not None:
    plt.scatter([MODEL_SIZE_B], [lora_accv], marker='*', s=320, color='black', label='Longformer-base + LoRA')
    plt.text(MODEL_SIZE_B * 1.15, lora_accv + 0.004, 'Longformer-base + LoRA', fontsize=10)
    print('Longformer LoRA AccV:', lora_accv)
else:
    print('No LoRA clean output found; run cells 4-5 first.')

plt.xscale('log')
plt.xlabel('Model Size')
plt.ylabel('Accuracy (Vector)')
y_max = 0.40
if lora_accv is not None:
    y_max = max(y_max, float(lora_accv) + 0.03)
plt.ylim(0.08, y_max)
plt.xticks([0.1, 1, 10, 100, 1000], ['100M', '1B', '10B', '100B', '1000B'])
plt.title('Figure 2 Baseline + Longformer LoRA Result')
plt.legend(loc='lower left')
plt.tight_layout()

plt.savefig(FIGURE_PATH, dpi=200, bbox_inches='tight')
plt.show()
print('Saved figure:', FIGURE_PATH)

